In [ ]:
import numpy as np
import psycopg2
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

In [ ]:

team_codes = {
            1 : "NJD", 2 : "NYI", 3 : "NYR", 4 : "PHI", 5 : "PIT",
            6 : "BOS", 7 : "BUF", 8 : "MTL", 9 : "OTT", 10 : "TOR",
            11: "ATL", 12 : "CAR", 13 : "FLA", 14 : "TBL", 15 : "WSH",
            16 : "CHI", 17 : "DET", 18 : "NSH", 19 : "STL", 20 : "CGY",
            21 : "COL", 22 : "EDM", 23 : "VAN", 24 : "ANA", 25 : "DAL",
            26 : "LAK", 27 : "PHX", 28 : "SJS", 29 : "CBJ", 30 : "MIN",
            31 : "TBL", 32 : "QUE", 33 : "WIN", 34 : "NSH", 35 : "CLR",
            36 : "SEN", 37 : "HAM", 38 : "PIR", 39 : "QUA", 40 : "DCG",
            41 : "MWN", 42 : "QBD", 43 : "MMR", 44 : "NYA", 45 : "SLE",
            46 : "OAK", 47 : "AFM", 48 : "KCS", 49 : "CLE", 50 : "DFL",
            51 : "BRK", 52 : "WPG", 53 : "ARI", 54 : "VGK", 55 : "SEA",
            56 : "CGD", 57 : "TAN", 58: "TSP", 59 : "UTA"
}

team_names = {y: x for x, y in team_codes.items()}


def get_games(year):
    hostname = 'database-1.c52o66wwou5d.us-east-2.rds.amazonaws.com'
    database = 'Sports'
    username = 'postgres'
    pwd = 'CS465Project1-4*'
    port_id = 5432

    conn = None
    crsr = None

    try:
        conn = psycopg2.connect(
            host = hostname,
            dbname = database,
            user = username,
            password = pwd,
            port = port_id
        )

        crsr = conn.cursor()
        crsr.execute(f"SELECT home_team_id, away_team_id, home_win FROM Games Where season_id = {year}{year+1}")
        ret = crsr.fetchall()
        conn.commit()
        #print(ret)
        return ret
    except Exception as error:
        print(error)
    finally:
        if crsr is not None:
            crsr.close()
        if conn is not None:
            conn.close()
            #print('Database connection terminated')

def db_request(query):
  hostname = 'database-1.c52o66wwou5d.us-east-2.rds.amazonaws.com'
  database = 'Sports'
  username = 'postgres'
  pwd = 'CS465Project1-4*'
  port_id = 5432

  conn = None
  crsr = None

  try:
      conn = psycopg2.connect(
          host = hostname,
          dbname = database,
          user = username,
          password = pwd,
          port = port_id
      )

      crsr = conn.cursor()
      crsr.execute(query)
      ret = crsr.fetchall()
      conn.commit()
      return ret
  except Exception as error:
      print(error)
  finally:
      if crsr is not None:
          crsr.close()
      if conn is not None:
          conn.close()

team_stats_dict = {}

def get_team_data(home_team, away_team, year):

  if team_stats_dict.get(f"{year}{home_team}") is None:
    home_season_query = f'''Select gamesplayed, xgoalsfor, xgoalsagainst, shotsongoalfor, xshotattmeptspercentage, shotsongoalagainst, pppercentage, pkpercentage
          from teamseasonstats where season_team_id = {year}{year+1}{home_team}
          '''
    h_data = db_request(home_season_query)
    h_data = h_data[0]
    team_stats_dict[f"{year}{home_team}"] = h_data
  else:
    h_data = team_stats_dict.get(f"{year}{home_team}")

  if team_stats_dict.get(f"{year}{away_team}") is None:
    away_season_query = f'''Select gamesplayed, xgoalsfor, xgoalsagainst, shotsongoalfor, xshotattmeptspercentage, shotsongoalagainst, pppercentage, pkpercentage
          from teamseasonstats where season_team_id = {year}{year+1}{away_team}
          '''
    a_data = db_request(away_season_query)
    a_data = a_data[0]
    team_stats_dict[f"{year}{away_team}"] = a_data
  else:
    a_data = team_stats_dict.get(f"{year}{away_team}")


  h_gf = int(h_data[0] * h_data[1])
  h_ga = int(h_data[0] * h_data[2])

  a_gf = int(a_data[0] * a_data[1])
  a_ga = int(a_data[0] * a_data[2])

  row = [h_gf, h_gf / (h_gf + h_ga), h_ga, h_data[3], h_data[4], h_ga / h_data[5], h_data[4] + (h_ga / h_data[5]), h_data[6], h_data[7],
         a_gf, a_gf / (a_gf + a_ga), a_ga, a_data[3], a_data[4], a_ga / a_data[5], a_data[4] + (a_ga / a_data[5]), a_data[6], a_data[7]]

  return np.array(row)

def main():

  X = None
  y = []

  count = 0
  for i in range(2014, 2020):
    count = 0
    for game in get_games(year):
      if count % 100 == 0:
        print(f"Processed {count} games...")
        print(team_stats_dict)

      home_id = int(str(game[0])[8:])
      away_id = int(str(game[1])[8:])

      all_stats = get_team_data(home_id, away_id, 2014)

      if X is None:
        X = all_stats
      else:
        X = np.vstack([X, all_stats])

      y = np.append(y, game[2])
      count += 1

  np_y = np.array(y)

  np.save('All_Game_Stats.npy', X)
  np.save('All_Game_Results.npy', np_y)

  #X = np.load('Game_Stats_2014.npy')
  #y = np.load('Game_Results_2014.npy')

  #X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 0)

  neigh = KNeighborsClassifier(n_neighbors=15)


  neigh.fit(X, y)

  # Get a game from 2022 for testing
  rand_game = get_team_data(53, 16, 2022)
  print(X)
  print(neigh.predict_proba(rand_game.reshape(1, -1)))

main()

[[2.08000000e+02 5.09803922e-01 2.00000000e+02 ... 5.86121264e-01
  2.10937000e-01 7.81954000e-01]
 [2.05000000e+02 4.44685466e-01 2.56000000e+02 ... 5.58489109e-01
  1.48148000e-01 8.63813000e-01]
 [2.36000000e+02 5.26785714e-01 2.12000000e+02 ... 5.80467606e-01
  1.84873000e-01 8.79699000e-01]
 ...
 [2.08000000e+02 4.83720930e-01 2.22000000e+02 ... 6.16296233e-01
  1.49425000e-01 8.48341000e-01]
 [1.64000000e+02 3.81395349e-01 2.66000000e+02 ... 6.03496705e-01
  1.31914000e-01 8.43065000e-01]
 [2.35000000e+02 5.17621145e-01 2.19000000e+02 ... 6.00858771e-01
  1.46551000e-01 7.85388000e-01]]
[[0.4 0.6]]
